In [ ]:
from src.ml import FEATURE_COLUMNS
from src.ml.retrain_gate import decide_retrain

dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("ml_schema", "ml", "ML schema")
dbutils.widgets.text("registered_model_name", "healthcare.ml.claim_denial_model", "Registered model name")
dbutils.widgets.text("champion_alias", "champion", "Champion alias")

catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
ml_schema = dbutils.widgets.get("ml_schema").strip()
registered_model_name = dbutils.widgets.get("registered_model_name").strip()
champion_alias = dbutils.widgets.get("champion_alias").strip()
gold_table = f"{catalog}.{gold_schema}.claim_features"

In [ ]:
decision = decide_retrain(
    spark,
    gold_table=gold_table,
    feature_columns=list(FEATURE_COLUMNS),
    registered_model_name=registered_model_name,
    champion_alias=champion_alias,
)

spark.createDataFrame([
    {
        "decided_at": None,
        "should_retrain": str(decision.should_retrain).lower(),
        "reason": decision.reason,
        "current_row_count": decision.current_row_count,
        "current_gold_version": decision.current_gold_version,
        "current_fingerprint": decision.current_fingerprint,
        "champion_run_id": decision.champion_run_id,
    }
]).selectExpr(
    "current_timestamp() AS decided_at",
    "should_retrain",
    "reason",
    "current_row_count",
    "current_gold_version",
    "current_fingerprint",
    "champion_run_id"
).write.mode("append").saveAsTable(f"{catalog}.{ml_schema}.retrain_decisions")

dbutils.jobs.taskValues.set(key="should_retrain", value=str(decision.should_retrain).lower())
dbutils.jobs.taskValues.set(key="reason", value=decision.reason)
dbutils.jobs.taskValues.set(key="current_training_row_count", value=str(decision.current_row_count))
dbutils.jobs.taskValues.set(key="current_gold_version", value=str(decision.current_gold_version))
dbutils.jobs.taskValues.set(key="current_data_fingerprint", value=decision.current_fingerprint)
print(decision.summary_line())